In [0]:
%sql
create table batch4.class.src_emp(
    ID INT,
    Name String,
    Salary INT,
    Location String,
    DeptID INT,
    LastUpdateDate TIMESTAMP
)
USING DELTA

In [0]:
%sql
create table batch4.class.etl_config(
    id int,
    tablename string,
    lastupdatedate timestamp,
    createddate timestamp,
    updateddate timestamp
)

In [0]:
%sql
insert into batch4.class.etl_config values (1,'src_emp','1900-01-01T00:01:01.200+00:00',current_timestamp(),current_timestamp())

In [0]:
%sql
select * from batch4.class.etl_config

In [0]:
%sql
insert into batch4.class.src_emp values 
(1,'John',5000,'New York',1,current_timestamp()),
(2,'Mary',6000,'New Jersey',2,current_timestamp()),
(3,'Mike',7000,'California',1,current_timestamp())

In [0]:
%sql
Update batch4.class.src_emp
set Location = 'Texas',
    LastUpdateDate = current_timestamp()
where ID = 1

In [0]:
%sql
insert into batch4.class.src_emp values 
(4,'Raj',5000,'Mumbai',1,current_timestamp()),
(5,'Ram',6000,'Delhi',2,current_timestamp()),
(6,'Ravi',7000,'Hyderabad',1,current_timestamp())

In [0]:
%sql
select * from batch4.class.src_emp

In [0]:
%sql
create table batch4.class.dim_emp (
    ID INT,
    Name String,
    Salary INT,
    Location String,
    DeptID INT,
    Hashbyte STRING,
    CreatedDate TIMESTAMP,
    UpdatedDate timestamp
)

In [0]:
from pyspark.sql.functions import sha2, concat_ws, current_timestamp

df = spark.sql("""select * from batch4.class.src_emp
                where LastUpdateDate >= (select Lastupdatedate from batch4.class.etl_config where tablename = 'src_emp')""")

trans_df = df.withColumn("Hashbyte", sha2(concat_ws(':', *df.columns), 256)) \
             .withColumn("CreatedDate", current_timestamp()) \
             .withColumn("UpdatedDate", current_timestamp()) \
             .drop("LastUpdateDate")

display(trans_df)

In [0]:
from delta.tables import DeltaTable

cols_to_insert = sorted([c for c in trans_df.columns if c != 'UpdatedDate'])
cols_to_update = sorted([c for c in trans_df.columns if c != 'CreatedDate'])

deltaTable = DeltaTable.forName(spark, "batch4.class.dim_emp")

deltaTable.alias("t") \
    .merge(trans_df.alias("s"), "s.ID = t.ID") \
        .whenMatchedUpdate(condition = "s.Hashbyte <> t.Hashbyte", set = {c: f"s.{c}" for c in cols_to_update}) \
        .whenNotMatchedInsert(values = {c: f"s.{c}" for c in cols_to_insert}) \
    .execute()

In [0]:
%sql
update batch4.class.etl_config
set lastupdatedate = (select max(lastupdatedate) + INTERVAL 1 SECOND from batch4.class.src_emp),
    updateddate = current_timestamp()
where tablename = 'src_emp'

In [0]:
%sql
select * from batch4.class.dim_emp

In [0]:
%sql
DESC HISTORY batch4.class.dim_emp